# ESM-2 35M — Full Training-Set Scoring (Colab)

> **هذا الـnotebook مكتوب عشان يشتغل على Google Colab بدون ما يحتاج منك تعديل.**

## ما الهدف؟

نحسب `esm2_llr` لكل ١٩٥ ألف missense variant في مجموعة التدريب + val + test.
هذه الميزة الجديدة راح تُضاف كعمود في الـparquet splits، وبعدين XGBoost يتدرّب مرة ثانية معها.

## الخطوات اللي راح تعملها أنت

1. افتح هذا النوتبوك في Colab: **File → Open notebook → GitHub → RayanAlDwlah/Genetic-Mutation-Detection-project**
2. **Runtime → Change runtime type → L4 GPU** (يحتاج Colab Pro $10/شهر)
   - أو **T4 GPU** (مجاني، بس أبطأ ~٢x)
3. **Runtime → Run all** (أو خلية بخلية)
4. لما تطلب خلية 2 إذن الوصول لـDrive، وافق (اختر حساب عنده صلاحية على مجلد `missense_esm2`)
5. لما تطلب خلية 3 GitHub PAT، الصقه (من Colab → Secrets → اسم السر `GH_PAT`)

### الوقت المتوقع

| GPU | الوقت الكلي | الاستهلاك (من 100 unit) |
|-----|-------------|-------------------------|
| L4  | ~٤–٥ ساعات | ~٢٢ unit |
| T4  | ~٩–١٠ ساعات | ~١٦ unit |
| A100 | ~٢–٣ ساعات | ~٢٤ unit |

## لو انقطعت الجلسة أو فشل VEP

ما في مشكلة — الـcache على Drive يحفظ:
- VEP: كل ٥ batches (~١٠٠٠ variant)، ومع exponential-backoff retry يصل لـ٦٠ ثانية
- Sequences: كل ١٠٠ transcript
- ESM-2 scores: كل ٥٠٠ variant

الـscript الجديد **صامد ضد فشل VEP REST**: لو batch فشل بعد ٨ محاولات، يتجاوزه ويكمل الباقي. الـvariants الفاشلة تبقى في cache-miss queue وتُعاد لاحقاً.

**شغّل كل الخلايا من البداية مرة ثانية**، راح يكمل من آخر checkpoint. **مهم**: تأكد إن خلية ٢ موصّلة بنفس الحساب اللي فيه مجلد `missense_esm2`.

## ١. تثبيت المتطلبات

In [ ]:
# Colab already has torch/transformers; we upgrade + add what's missing.
%pip install -q --upgrade transformers
%pip install -q pyarrow requests

import torch
assert torch.cuda.is_available(), (
    'No GPU! Runtime → Change runtime type → L4 GPU (Pro) or T4 GPU (free)'
)
print('GPU:', torch.cuda.get_device_name(0))

## ٢. Mount Google Drive (للـcache)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CACHE_DIR = '/content/drive/MyDrive/missense_esm2/cache'
!mkdir -p {CACHE_DIR}
print('Cache dir:', CACHE_DIR)

## ٣. Clone repo من GitHub

نستخدم الـPAT من Colab Secrets. للإعداد:
1. في Colab: اضغط على 🔑 (يسار)
2. Add new secret
3. Name: `GH_PAT` — Value: توكنك (الذي عنده `repo` + `workflow` scope)
4. فعّل "Notebook access"

In [ ]:
import os
from google.colab import userdata
GH_PAT = userdata.get('GH_PAT')
REPO = 'RayanAlDwlah/Genetic-Mutation-Detection-project'
!rm -rf /content/repo && git clone https://{GH_PAT}@github.com/{REPO}.git /content/repo
%cd /content/repo
!git log --oneline -3

## ٤. تكوين scorer + نسخ الـcache إلى مكان الـrepo

`src/esm2_scorer.py` يعمل resumable cache في `data/intermediate/esm2/`. نربطه بـDrive عن طريق symlink.

In [ ]:
import os

os.makedirs('/content/repo/data/intermediate', exist_ok=True)
local_link = '/content/repo/data/intermediate/esm2'

# Nuke any stale link or directory from a previous session.
if os.path.islink(local_link):
    os.remove(local_link)
elif os.path.isdir(local_link):
    import shutil
    shutil.rmtree(local_link)

# Create symlink: data/intermediate/esm2 → Drive cache
os.symlink(CACHE_DIR, local_link)

# ── VERIFY the symlink actually points to Drive ──
assert os.path.islink(local_link), (
    f'❌ Symlink creation failed! {local_link} is not a symlink.'
)
target = os.readlink(local_link)
assert target == CACHE_DIR, (
    f'❌ Symlink points to {target!r} instead of {CACHE_DIR!r}'
)
assert os.path.isdir(local_link), (
    f'❌ Symlink target does not exist — is Drive mounted? '
    f'Check cell 2 and re-authorize if needed.'
)

print(f'✅ Symlink OK: {local_link} → {CACHE_DIR}')
!ls -la /content/repo/data/intermediate/esm2/ 2>&1 | head -10

## ٥. سكور للـsplits الثلاثة

على T4: ~٠.١٥ ثانية/variant.
- train: ١٩٥K × ٠.١٥ = ~٨ ساعات
- val: ٢٨K × ٠.١٥ = ~١ ساعة
- test: ٢٨K × ٠.١٥ = ~١ ساعة

**المجموع: ~١٠ ساعات**. Colab free tier يسمح ~١٢ ساعة قبل ما يفصل. فيه احتمال يحتاج جلستين.

Cache على Drive محفوظ كل ٥٠٠ variant.

In [ ]:
import os
import sys

sys.path.insert(0, '/content/repo')

# ── Pre-flight checks ──
_link = '/content/repo/data/intermediate/esm2'
assert os.path.islink(_link), (
    f'❌ {_link} is not a symlink. Go run cell 8 first!'
)
assert os.readlink(_link) == CACHE_DIR, (
    f'❌ Symlink points to wrong target. Re-run cell 8.'
)

# Verify the resilient VEP retry fix is present in the cloned code.
_scorer_src = open('/content/repo/src/esm2_scorer.py').read()
assert 'MAX_BACKOFF_SEC' in _scorer_src and 'failed_batches' in _scorer_src, (
    '❌ Stale code — missing resilient VEP retry fix. '
    'Re-run cell 6 (git clone) to get latest main.'
)
print('✅ Symlink + code version OK — starting scoring\n')

import pandas as pd
from src.esm2_scorer import score_variants

for split_name in ['val', 'test', 'train']:
    print(f'\n{"=" * 60}\n[{split_name}] scoring…\n{"=" * 60}')
    df = pd.read_parquet(f'/content/repo/data/splits/{split_name}.parquet')
    query = df[['variant_key', 'chr', 'pos', 'ref', 'alt']].drop_duplicates('variant_key')
    print(f'  {len(query):,} unique variants to score')
    scored = score_variants(query)
    out_path = f'/content/repo/data/intermediate/esm2/scores_{split_name}.parquet'
    scored.to_parquet(out_path, index=False)
    print(f'  [done] {split_name}: {scored["esm2_llr"].notna().sum():,} / {len(scored):,} scored')

print('\n\n✅ ESM-2 scoring complete.')

## ٦. Push النتايج على GitHub

هذه الخلية تأخذ الـparquets الجديدة وتدفعها على `main` مباشرة. استخدام GH_PAT اللي عنده `repo` + `workflow` scopes.

In [ ]:
%cd /content/repo
!git config user.email 'colab@automation'
!git config user.name 'Colab Automation'
# Only commit the score parquets (they're symlinked to Drive so git sees them as regular files).
!cp /content/drive/MyDrive/missense_esm2/cache/scores_*.parquet data/intermediate/esm2/
!git add data/intermediate/esm2/scores_train.parquet data/intermediate/esm2/scores_val.parquet data/intermediate/esm2/scores_test.parquet
!git commit -m 'Stage 2.1 ESM-2: full-training-set zero-shot LLR scoring (Colab T4)'
!git push origin HEAD:main

## ٧. التالي

الآن على جهازي المحلي، أنا راح أسحب الـparquets الجديدة وأعيد تدريب XGBoost مع العمود الجديد `esm2_llr`. النتيجة راح تنزل تلقائياً على `main`.

### لو طلعت مشكلة

- **`No GPU`**: Runtime → Change runtime type → L4 أو T4
- **خطأ Drive**: اعمل re-authorize في الخلية ٢، وتأكد إنك تختار حساب عنده صلاحية على مجلد `missense_esm2`
- **`Symlink creation failed` في خلية ٨**: غالبًا Drive مو موصّل صح. رجّع خلية ٢ وأعد الربط
- **`Stale code — missing VEP checkpoint fix`**: رجّع شغّل خلية ٦ (git clone) عشان يسحب آخر نسخة
- **خطأ VEP REST (`502`/`503`)**: انتظر ١٠ دقائق ثم شغّل مرة ثانية
- **انقطاع الجلسة**: شغّل كل الخلايا من ١ إلى ١٠ بالترتيب — الـcache في Drive محفوظ

### نقطة مهمة عند تبديل الحسابات

لو اضطريت تبدّل حساب Google (بسبب usage limits)، **لازم** تعيد تشغيل كل الخلايا من ١ إلى ١٠. الـ symlink ومسار Drive يعتمدون على الجلسة الحالية — لو شغّلت خلية ١٠ بدون ما تعيد ١→٨، شغلك بيضيع.

أي مشكلة ثانية أرسلها لي.